# Pontificia Universidad Javeriana

**Student:** Alisson Garcia Salazar  
**Date:** September 9, 2026  
**Topic:** Introduction to Apache Spark and multiclass classification

## Apache Spark: Beginner Tutorial with the Iris Dataset

### Objective

In this notebook, follow the tutorial to learn how to use PySpark to build a basic machine learning workflow. The goal is to classify Iris flowers into three species from four measurements and compare the performance of three algorithms: Decision Tree, Random Forest, and Naive Bayes.

This is a supervised multiclass classification problem because each training observation has a known species and there are three possible output classes.

### Variables

| Type | Variable | Meaning |
|---|---|---|
| Target | `Species` | Actual flower species |
| Predictor | `SepalLengthCm` | Sepal length in centimeters |
| Predictor | `SepalWidthCm` | Sepal width in centimeters |
| Predictor | `PetalLengthCm` | Petal length in centimeters |
| Predictor | `PetalWidthCm` | Petal width in centimeters |
| Identifier | `Id` | Row identifier; it is not used for prediction |

### Workflow

1. Install and import the required libraries.
2. Create a Spark session.
3. Load and explore the dataset.
4. Convert the species from text to numeric labels.
5. Combine the predictors into a feature vector.
6. Scale the features.
7. Split the data into training and test sets.
8. Train, evaluate, and compare three models.


In [1]:
# Install PySpark, the Python interface for Apache Spark.
# --quiet reduces installation messages and keeps the notebook output cleaner.
!pip install pyspark --quiet


In [2]:
# Install tabulate to display the final comparison as a table.
!pip install tabulate


## 1. Import Libraries

PySpark makes it possible to work with distributed data through Python. Although this dataset contains only 150 records and could be processed locally, it is useful for learning a workflow that can later scale to much larger datasets.

The original tutorial imports some tools that are not directly used in the executed workflow, such as `numpy`, `pandas`, `train_test_split`, `Pipeline`, and `VectorAssembler`. I keep them to respect the original exercise, but the main components used here are `SparkSession`, the three classifiers, `StringIndexer`, `StandardScaler`, `DenseVector`, and the multiclass evaluator.


In [3]:
# General-purpose libraries for numerical computing and data manipulation.
import numpy as np
import pandas as pd

# Main Apache Spark components.
import pyspark
from pyspark.sql import SparkSession

# The three classification algorithms compared in this notebook.
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, NaiveBayes

# Evaluator for classification problems with more than two classes.
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Tools for preparing and transforming features.
from pyspark.ml.feature import StandardScaler, StringIndexer, VectorAssembler, VectorIndexer, OneHotEncoder

# Pipeline can chain transformations and models in more complete projects.
from pyspark.ml import Pipeline

# DenseVector combines the four flower measurements into a single feature column.
from pyspark.ml.linalg import DenseVector

# Scikit-learn utilities included in the original tutorial.
import sklearn
from sklearn.model_selection import train_test_split

# Utilities for tabular presentation and memory cleanup.
from tabulate import tabulate
import gc


## 2. Build Spark Session

`SparkSession` is the entry point for creating DataFrames and using Spark SQL and Spark ML. `getOrCreate()` prevents duplicate sessions: if a compatible session already exists, Spark reuses it.

The memory and core settings describe the resources requested for execution. In a local or academic environment, Spark may adapt its behavior to the resources that are actually available.


In [4]:
# Create a Spark session, or reuse one if it already exists.
# appName identifies the application in Spark logs.
# These settings request 1 GB per executor and 4 CPU cores.
spark = (SparkSession.builder
         .appName('Apache Spark Beginner Tutorial')
         .config("spark.executor.memory", "1G")
         .config("spark.executor.cores", "4")
         .getOrCreate())


In [5]:
# INFO displays detailed messages about Spark's internal execution.
# WARN could be used for shorter output, but INFO is kept for this exercise.
spark.sparkContext.setLogLevel('INFO')


In [6]:
# Check the Spark version used to run the exercise.
spark.version


'4.0.4'

## 3.Load Data

The CSV file is loaded as a distributed DataFrame. `header=true` prevents the column names from being treated as an observation, while `inferSchema=true` allows Spark to recognize `Id` as an integer and the measurements as decimal values.



In [7]:
# Iris.csv is a relative path, so Spark looks for it in the current working directory.
url = 'Iris.csv'

# Read the CSV, use the first row as the header, and infer each column's data type.
data = (spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(url))

# Ask Spark to retain this DataFrame for faster reuse.
# Spark evaluates lazily, so the cache is materialized by an action such as count().
data.cache()


DataFrame[Id: int, SepalLengthCm: double, SepalWidthCm: double, PetalLengthCm: double, PetalWidthCm: double, Species: string]

In [8]:
# This check uses an absolute path (/Iris.csv), which is different from 'Iris.csv'.
# It may return False even when the previous relative-path load works correctly.
import os
print(os.path.exists('/Iris.csv'))


False


## 4. Data Exploration & Preparation
I explored the dataset to understand its structure and verify that the data was loades correctly.
I checked the number of records, column names, and data types, reviewed sample observations, analyzed the distribution of the three species.
Also converted the categorical species column into a numerical label because Spark ML classification algorithms require the target variable to be numeric.

In [9]:
# count() is a Spark action: it executes the plan and returns the total number of rows.
data.count()


150

In [10]:
# Review column names, data types, and whether the columns can contain null values.
data.printSchema()


root
 |-- Id: integer (nullable = true)
 |-- SepalLengthCm: double (nullable = true)
 |-- SepalWidthCm: double (nullable = true)
 |-- PetalLengthCm: double (nullable = true)
 |-- PetalWidthCm: double (nullable = true)
 |-- Species: string (nullable = true)



In [11]:
# Display five observations to visually verify that the data loaded correctly.
data.show(5)


+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|
+---+-------------+------------+-------------+------------+-----------+
only showing top 5 rows


In [12]:
# Group by species to check the balance of the target variable.
data.groupBy('species').count().show()


+---------------+-----+
|        species|count|
+---------------+-----+
| Iris-virginica|   50|
|    Iris-setosa|   50|
|Iris-versicolor|   50|
+---------------+-----+



In [13]:
# Obtain count, mean, standard deviation, minimum, and maximum values.
# Species is text, so its mean and standard deviation appear as NULL.
data.describe().show()


+-------+------------------+------------------+-------------------+------------------+------------------+--------------+
|summary|                Id|     SepalLengthCm|       SepalWidthCm|     PetalLengthCm|      PetalWidthCm|       Species|
+-------+------------------+------------------+-------------------+------------------+------------------+--------------+
|  count|               150|               150|                150|               150|               150|           150|
|   mean|              75.5| 5.843333333333335| 3.0540000000000007|3.7586666666666693|1.1986666666666672|          NULL|
| stddev|43.445367992456916|0.8280661279778637|0.43359431136217375| 1.764420419952262|0.7631607417008414|          NULL|
|    min|                 1|               4.3|                2.0|               1.0|               0.1|   Iris-setosa|
|    max|               150|               7.9|                4.4|               6.9|               2.5|Iris-virginica|
+-------+------------------+----

### Interpretation of the Exploratory Analysis

- The dataset has **150 records**, and each column reports 150 values, so no missing values are apparent in this summary.
- Each species has **50 flowers**. The dataset is perfectly balanced, which makes accuracy a reasonable initial metric.
- The four measurements use centimeters but have different levels of dispersion. For example, petal length has a standard deviation of approximately 1.76, while sepal width has a standard deviation of approximately 0.43.
- `Id` ranges from 1 to 150 and serves only as an identifier. Including it as a predictor could introduce an artificial pattern related to the order of the file.
- The mean of `Species` appears as `NULL` because an average cannot be calculated for text values.


### Converting the Target Variable

Spark ML algorithms expect a numeric label. `StringIndexer` learns a mapping between the three text values and numeric codes such as `0.0`, `1.0`, and `2.0`. These numbers are **codes**, not quantities: a species encoded as 2 is not “greater than” a species encoded as 1.


In [14]:
# StringIndexer learns a mapping between each species and a numeric code.
SIndexer = StringIndexer(inputCol='species', outputCol='species_indx')
data = SIndexer.fit(data).transform(data)

# Verify that species_indx was added without removing the original species column.
data.show(5)


+---+-------------+------------+-------------+------------+-----------+------------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|species_indx|
+---+-------------+------------+-------------+------------+-----------+------------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|         0.0|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|         0.0|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|         0.0|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|         0.0|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|         0.0|
+---+-------------+------------+-------------+------------+-----------+------------+
only showing top 5 rows


**Observed example:** `Iris-setosa` was encoded as `0.0`. The original text column is retained, which makes it possible to verify the relationship between each species and its numeric code.


## 5. Feature Engineering

Spark ML normally expects two columns:

- `label`: the known response that the model must learn to predict.
- `features`: one vector containing all predictor variables.

No new variables are created in this exercise. Instead, the four existing measurements are reorganized into the format required by the models.


In [15]:
# Select the label and the four measurements that describe flower morphology.
# Id is excluded because it only identifies rows and provides no biological information.
df = data.select(
    "species_indx",
    "SepalLengthCm", "SepalWidthCm",
    "PetalLengthCm", "PetalWidthCm"
)

df.show(5)


+------------+-------------+------------+-------------+------------+
|species_indx|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|
+------------+-------------+------------+-------------+------------+
|         0.0|          5.1|         3.5|          1.4|         0.2|
|         0.0|          4.9|         3.0|          1.4|         0.2|
|         0.0|          4.7|         3.2|          1.3|         0.2|
|         0.0|          4.6|         3.1|          1.5|         0.2|
|         0.0|          5.0|         3.6|          1.4|         0.2|
+------------+-------------+------------+-------------+------------+
only showing top 5 rows


In [16]:
# Convert each row into a tuple: (label, feature vector).
# Example: (0.0, [5.1, 3.5, 1.4, 0.2]).
input_data = df.rdd.map(lambda x: (x[0], DenseVector(x[1:])))


In [17]:
# Create the standard format expected by Spark ML algorithms:
# one label column and one vector-valued features column.
df_indx = spark.createDataFrame(input_data, ["label", "features"])


In [18]:
# Verify that the four measurements were combined into features.
df_indx.show(5)


+-----+-----------------+
|label|         features|
+-----+-----------------+
|  0.0|[5.1,3.5,1.4,0.2]|
|  0.0|[4.9,3.0,1.4,0.2]|
|  0.0|[4.7,3.2,1.3,0.2]|
|  0.0|[4.6,3.1,1.5,0.2]|
|  0.0|[5.0,3.6,1.4,0.2]|
+-----+-----------------+
only showing top 5 rows


### Transformation Example

The first flower changes from four separate columns into this vector:

```text
label = 0.0
features = [5.1, 3.5, 1.4, 0.2]
```

The order of the vector is important: sepal length, sepal width, petal length, and petal width. Changing this order between training and prediction would change the meaning of the data.


## 6. Feature Scaling

`StandardScaler` divides each feature by its standard deviation without subtracting the mean. This makes the measurements comparable while keeping them non-negative for Naive Bayes. The same scaled data is used to compare all three models.

In [19]:
# StandardScaler uses the standard deviation of each variable.
# By default, withStd=True and withMean=False: it divides by the standard deviation
# but does not subtract the mean.
stdScaler = StandardScaler(inputCol="features", outputCol="features_scaled")

# fit calculates the scaling parameters from the data.
scaler = stdScaler.fit(df_indx)

# transform applies those parameters and creates features_scaled.
df_scaled = scaler.transform(df_indx)


In [20]:
# Compare the original vector with its scaled version.
df_scaled.show(5)


+-----+-----------------+--------------------+
|label|         features|     features_scaled|
+-----+-----------------+--------------------+
|  0.0|[5.1,3.5,1.4,0.2]|[6.15892840883878...|
|  0.0|[4.9,3.0,1.4,0.2]|[5.9174018045706,...|
|  0.0|[4.7,3.2,1.3,0.2]|[5.67587520030241...|
|  0.0|[4.6,3.1,1.5,0.2]|[5.55511189816831...|
|  0.0|[5.0,3.6,1.4,0.2]|[6.03816510670469...|
+-----+-----------------+--------------------+
only showing top 5 rows


In [21]:
# Drop features so the models explicitly use features_scaled.
df_scaled = df_scaled.drop("features")


> **Methodological observation:** in this tutorial, the scaler is fitted before the data is split. In a rigorous predictive project, the split should happen first and the scaler should be fitted only on the training data. This prevents statistical information from the test set from influencing data preparation. A Spark `Pipeline` would also help enforce the correct order.


## 7. Splitting the Data

The training set is used to learn the parameters of each algorithm. The test set is kept outside that fitting process and is used to estimate performance on previously unseen observations.

`randomSplit([0.9, 0.1])` produces approximate proportions; it does not guarantee exactly 135 and 15 records. The seed `12345` makes the random assignment reproducible in the same environment.


In [22]:
# Randomly split the data into approximately 90% training and 10% test data.
# The seed makes the same assignment reproducible in the same environment.
train_data, test_data = df_scaled.randomSplit([0.9, 0.1], seed=12345)


In [23]:
# Inspect examples from the dataset used to train the models.
train_data.show(5)


+-----+--------------------+
|label|     features_scaled|
+-----+--------------------+
|  0.0|[5.19282199176603...|
|  0.0|[5.31358529390013...|
|  0.0|[5.31358529390013...|
|  0.0|[5.31358529390013...|
|  0.0|[5.43434859603422...|
+-----+--------------------+
only showing top 5 rows


## 8. Building, Training, and Evaluating the Models

All three models receive the same label and scaled-feature columns, making the procedure comparable. Each model follows the same sequence:

1. Instantiate the algorithm and define its parameters.
2. Run `fit(train_data)` to train it.
3. Run `transform(test_data)` to generate predictions.
4. Calculate accuracy with the same evaluator.

Accuracy is defined as:

$$Accuracy=\frac{\text{correct predictions}}{\text{total predictions}}$$


In [24]:
# Store model names and results in lists to build a consistent comparison table.
model = ['Decision Tree', 'Random Forest', 'Naive Bayes']
model_results = []


In [25]:
# --- Decision Tree Classifier ---
# It learns rules such as: if a measurement exceeds a threshold, follow one branch.
dtc = DecisionTreeClassifier(labelCol="label", featuresCol="features_scaled")
dtc_model = dtc.fit(train_data)               # Learn the rules from the training data.
dtc_pred = dtc_model.transform(test_data)     # Predict labels for the test data.

# Accuracy = number of correct predictions / total number of predictions.
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
dtc_acc = evaluator.evaluate(dtc_pred)
model_results.extend([[model[0], '{:.2%}'.format(dtc_acc)]])


### Decision Tree

The tree creates consecutive rules based on measurement thresholds. It is easy to interpret, but a single tree can change considerably when the training sample changes. The observed result was **90.91%**. This percentage is consistent with 10 correct predictions out of 11 test cases, meaning that one observation was probably misclassified.


In [26]:
# --- Random Forest Classifier ---
# Combine 10 trees and use their collective vote. This usually reduces the
# sensitivity that a single tree would have to small variations in the data.
rfc = RandomForestClassifier(
    labelCol="label", featuresCol="features_scaled", numTrees=10
)
rfc_model = rfc.fit(train_data)
rfc_pred = rfc_model.transform(test_data)

evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
rfc_acc = evaluator.evaluate(rfc_pred)
model_results.extend([[model[1], '{:.2%}'.format(rfc_acc)]])


### Random Forest

The forest combines the decisions of 10 trees built with variations in the samples and features. Their collective vote usually reduces the variance of a single Decision Tree. In this particular split, it achieved **100% accuracy**.


In [27]:
# --- Multinomial Naive Bayes ---
# Estimate class probabilities under a conditional-independence assumption.
# smoothing=1.0 applies Laplace smoothing to avoid probabilities equal to zero.
# Multinomial Naive Bayes requires non-negative features; this condition is met
# because StandardScaler did not center the data (withMean=False).
nbc = NaiveBayes(
    smoothing=1.0, modelType="multinomial",
    labelCol="label", featuresCol="features_scaled"
)
nbc_model = nbc.fit(train_data)
nbc_pred = nbc_model.transform(test_data)

evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
nbc_acc = evaluator.evaluate(nbc_pred)
model_results.extend([[model[2], '{:.2%}'.format(nbc_acc)]])


### Naive Bayes

Naive Bayes calculates probabilities for each class and assumes conditional independence among the features. `smoothing=1.0` prevents zero probabilities. The multinomial type was mainly designed for counts or frequencies; therefore, although it accepts these positive measurements and achieved **100% accuracy**, other probabilistic approaches would also be worth comparing for continuous measurements.


In [28]:
# Ask Python's garbage collector to release objects that are no longer needed.
# This does not delete the trained models or replace Spark's own memory management.
gc.collect()


680

## 9. Comparing the Results


In [29]:
# Display all results with consistent headings and percentage formatting.
print(tabulate(model_results, headers=["Classifier Models", "Accuracy"]))


Classifier Models    Accuracy
-------------------  ----------
Decision Tree        90.91%
Random Forest        100.00%
Naive Bayes          100.00%


### Final Interpretation

Random Forest and Naive Bayes tied in accuracy and outperformed the Decision Tree. However, this does not prove that they are perfect models: the test set is very small and represents only one random split. The 100% result describes this specific test, not guaranteed performance on new flowers.

### What I Learned

- Spark separates lazy transformations from actions that trigger computation.
- Text labels must be converted to numeric codes, and predictors must be combined into a feature vector.
- Data preparation must preserve the order and meaning of all variables.
- A fair model comparison requires the same data split and evaluation metric.
- Accuracy must be interpreted together with the size and composition of the test set.

### Possible Improvements

1. Split the data before fitting the scaler to prevent data leakage.
2. Include a confusion matrix to identify which species are confused.
3. Calculate per-class precision, recall, and F1-score in addition to accuracy.
4. Use cross-validation and tune the models' hyperparameters.
5. Combine indexing, scaling, and modeling in a reproducible Spark `Pipeline`.

With these considerations, the notebook provides a clear introduction to the complete classification workflow in Spark ML.
